# Lab 9 - Semantic Caching and Cost Control

**Production Readiness Pack | CPU | OpenAI API key optional**

In production, the cheapest and fastest LLM call is the one you do not make. But caching LLM answers has new risks: two questions can sound similar while requiring different answers.

## Learning Objectives

1. Compare exact caching with semantic caching.
2. Implement a local semantic cache using embeddings.
3. Measure latency and estimated token cost savings.
4. Tune similarity thresholds and identify false hits.
5. Decide which responses are safe to cache in production.

In [ ]:
!uv pip install -q openai sentence-transformers pandas scikit-learn

In [ ]:
import os
import time
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from google.colab import userdata
    if not os.environ.get("OPENAI_API_KEY"):
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
except Exception:
    pass

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
DOC_VERSION = "course-kb-v1"
CACHE_TTL_SECONDS = 60 * 60
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"OpenAI enabled: {USE_OPENAI}")

## Real-World Mental Model

Semantic caching sits between your app and your model call. It asks: **Have we already answered something with the same meaning under the same safety constraints?**

That last phrase matters. A cache key in an LLM app is rarely just the user question. In production, it may need to include:

- tenant or customer ID
- user permissions
- model name and prompt version
- document index version
- locale or region
- product plan or contract type

The aha moment in this lab is that semantic caching is both a performance tool and a correctness risk. A false cache hit is a fast wrong answer.

## 1. A Small Public Knowledge Base

We use stable public course facts because these are reasonable cache candidates. Later we add a danger case where similar wording should not share an answer.

In [ ]:
FAQ_ANSWERS = {
    "qlora": "QLoRA fine-tunes small LoRA adapters on top of a frozen 4-bit quantized base model, reducing GPU memory requirements.",
    "rag": "RAG retrieves relevant context at request time and injects it into the prompt so answers can be grounded in external documents.",
    "serving": "An OpenAI-compatible API lets the same SDK call different model backends by changing the base_url.",
    "premium_refund": "Premium plan refunds are available within 30 days of purchase.",
    "enterprise_refund": "Enterprise refund terms are governed by the signed contract and must be reviewed by the account team.",
}

def demo_answer(question: str):
    q = question.lower()
    if "enterprise" in q and "refund" in q:
        return FAQ_ANSWERS["enterprise_refund"]
    if "premium" in q and "refund" in q:
        return FAQ_ANSWERS["premium_refund"]
    if "qlora" in q or "fine-tun" in q or "gpu memory" in q or "gpu ram" in q:
        return FAQ_ANSWERS["qlora"]
    if "rag" in q or "retrieval" in q:
        return FAQ_ANSWERS["rag"]
    if "base_url" in q or "openai-compatible" in q or "serving" in q:
        return FAQ_ANSWERS["serving"]
    return "I do not have enough information to answer from the public FAQ."

def call_model(question: str):
    if not USE_OPENAI:
        time.sleep(0.25)
        return demo_answer(question)
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(model=MODEL_NAME, temperature=0, messages=[{"role": "system", "content": "Answer using concise course deployment knowledge."}, {"role": "user", "content": question}])
    return response.choices[0].message.content

def estimate_cost_usd(question: str, answer: str):
    input_tokens = max(1, len(question) // 4)
    output_tokens = max(1, len(answer) // 4)
    return round((input_tokens * 0.15 / 1_000_000) + (output_tokens * 0.60 / 1_000_000), 6)

## 2. Exact Cache Baseline

Exact caching is simple and reliable, but it only works when users repeat the same string.

In [ ]:
exact_cache: Dict[str, Dict] = {}

def ask_with_exact_cache(question: str):
    start = time.perf_counter()
    if question in exact_cache:
        entry = exact_cache[question]
        return {"strategy": "exact", "hit": True, "answer": entry["answer"], "latency_ms": round((time.perf_counter() - start) * 1000, 2), "estimated_cost": 0.0}
    answer = call_model(question)
    cost = estimate_cost_usd(question, answer)
    exact_cache[question] = {"answer": answer, "created_at": time.time(), "model": MODEL_NAME, "doc_version": DOC_VERSION}
    return {"strategy": "exact", "hit": False, "answer": answer, "latency_ms": round((time.perf_counter() - start) * 1000, 2), "estimated_cost": cost}

q1 = "How does QLoRA reduce GPU memory?"
q2 = "How does QLoRA reduce GPU memory?"
q3 = "Why does QLoRA need less GPU RAM?"
pd.DataFrame([ask_with_exact_cache(q) for q in [q1, q2, q3]])

## 3. Semantic Cache With Metadata

A production cache entry needs more than a question and answer. At minimum, include model name, document version, creation time, and a TTL.

In [ ]:
@dataclass
class CacheEntry:
    question: str
    answer: str
    embedding: np.ndarray
    created_at: float
    model_name: str
    doc_version: str
    estimated_cost: float

semantic_cache: List[CacheEntry] = []

def embed(text: str):
    return embedder.encode([text], normalize_embeddings=True)[0]

def is_fresh(entry: CacheEntry, ttl_seconds: int = CACHE_TTL_SECONDS):
    return (time.time() - entry.created_at) <= ttl_seconds

def lookup_semantic_cache(question: str, threshold: float, doc_version: str = DOC_VERSION):
    if not semantic_cache:
        return None, 0.0
    query_embedding = embed(question)
    candidates = []
    for entry in semantic_cache:
        if entry.doc_version != doc_version or entry.model_name != MODEL_NAME or not is_fresh(entry):
            continue
        score = float(cosine_similarity([query_embedding], [entry.embedding])[0][0])
        candidates.append((score, entry))
    if not candidates:
        return None, 0.0
    score, entry = max(candidates, key=lambda item: item[0])
    if score >= threshold:
        return entry, score
    return None, score

def ask_with_semantic_cache(question: str, threshold: float = 0.78):
    start = time.perf_counter()
    entry, score = lookup_semantic_cache(question, threshold=threshold)
    if entry:
        return {"strategy": "semantic", "hit": True, "similarity": round(score, 3), "matched_question": entry.question, "answer": entry.answer, "latency_ms": round((time.perf_counter() - start) * 1000, 2), "estimated_cost": 0.0}
    answer = call_model(question)
    cost = estimate_cost_usd(question, answer)
    semantic_cache.append(CacheEntry(question=question, answer=answer, embedding=embed(question), created_at=time.time(), model_name=MODEL_NAME, doc_version=DOC_VERSION, estimated_cost=cost))
    return {"strategy": "semantic", "hit": False, "similarity": round(score, 3), "matched_question": None, "answer": answer, "latency_ms": round((time.perf_counter() - start) * 1000, 2), "estimated_cost": cost}

## 4. Benchmark Semantic Cache Hits

The second and third questions are paraphrases. They should be reusable if the threshold is reasonable.

In [ ]:
semantic_cache.clear()
questions = ["How does QLoRA reduce GPU memory?", "Why does QLoRA need less GPU RAM?", "Explain how QLoRA makes fine-tuning cheaper on GPUs."]
results = [ask_with_semantic_cache(q, threshold=0.72) for q in questions]
pd.DataFrame(results)[["hit", "similarity", "matched_question", "latency_ms", "estimated_cost", "answer"]]

## 5. The Danger Case: False Cache Hits

Semantic similarity is not the same as business equivalence. These two questions sound close but require different answers.

In [ ]:
semantic_cache.clear()
danger_questions = ["What is the refund policy for the premium plan?", "What is the refund policy for the enterprise plan?"]
loose_results = [ask_with_semantic_cache(q, threshold=0.55) for q in danger_questions]
pd.DataFrame(loose_results)[["hit", "similarity", "matched_question", "answer"]]

If the enterprise question reused the premium answer, that is a **false hit**. In production this can be worse than a slow response because it gives a fast wrong answer.

In [ ]:
semantic_cache.clear()
strict_results = [ask_with_semantic_cache(q, threshold=0.88) for q in danger_questions]
pd.DataFrame(strict_results)[["hit", "similarity", "matched_question", "answer"]]

## 6. Cache Safety Rules

A good production cache policy often includes:

- Scope by tenant, user, permissions, locale, and product/version when answers differ by audience.
- Invalidate by document version or content hash when the knowledge base changes.
- Use TTLs for policies or prices that can change.
- Avoid caching personalized, private, legal, medical, or financial answers unless the cache key includes all required access-control context.

In [ ]:
pd.DataFrame([
    {"scenario": "Public FAQ: What is QLoRA?", "safe": True, "reason": "Stable public answer."},
    {"scenario": "User asks for their invoice total", "safe": False, "reason": "Personalized private data."},
    {"scenario": "Refund policy by plan", "safe": "Maybe", "reason": "Must scope by plan and document version."},
    {"scenario": "Latest incident status", "safe": False, "reason": "Time-sensitive and changes quickly."},
    {"scenario": "Product docs for version 2.1", "safe": True, "reason": "Safe if cache key includes docs version."},
])

## Gotchas Before Production

Semantic caching is attractive because it can reduce token spend immediately, but it changes system behavior:

| Gotcha | Why It Matters | Mitigation |
| --- | --- | --- |
| Similar wording, different policy | Premium and enterprise users may need different answers | Include plan/tenant/version in the cache scope |
| Stale documents | Cached answer may cite an old policy | Invalidate by document version or content hash |
| Personalized answers | One user's answer could leak to another user | Do not cache private responses unless scoped by user and permission |
| Model upgrades | A cached answer hides behavior changes from the new model | Include model and prompt version in metadata |
| Weak observability | You cannot tell whether a response was generated or cached | Log cache hit/miss, matched question, and similarity |

When in doubt, start with exact caching for safe public FAQs, then graduate to semantic caching for carefully scoped domains.

## Student Exercise

1. Try thresholds `0.60`, `0.75`, and `0.90` on the QLoRA questions.
2. Try the same thresholds on the refund questions.
3. Pick a threshold and explain the tradeoff.
4. Add one metadata field you would need for your own product, such as tenant ID, region, or document version.

Reflection: where would you place semantic caching in your Capstone architecture, if anywhere?

## Key Takeaways

- Exact caching is safe but misses paraphrases.
- Semantic caching can reduce latency and cost but introduces false-hit risk.
- Threshold tuning is a product and risk decision, not just a math decision.
- Metadata and invalidation rules are what make caching production-grade.